**!!! NOTEBOOK FOR USAGE REFERENCE !!!**

This notebook is not intended to provide actual code; Instead, it is just a "getting started", something you can take as inspiration when using this testbed.

For developers: please do not commit the outputs to the library. Always clear all outputs before saving.

----

Downloads, enrich, filter, save splits

Generates metadata_filtered

General notebook structure:
1. Inicial set: ~300 entities to be partially enriched
2. Enriched set: fully enriched and verified, including manual checks/inputs. Manually remove mismaches name<->pantheon.
3. Filtered set: filtered 100 entities based on representativenss balancing
4. Entities appear exactly in this order in the table representations.
5. Images should only be saved for the entities in the final enriched set


In [ ]:
import os
import sys
import dotenv
import pandas as pd
import scipy.io as sio
import torch.utils.checkpoint

import nltk
nltk.download('wordnet')
from nltk.corpus import wordnet as wn

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 30)

sys.path.append('../TRDP-unlearning')
dotenv.load_dotenv('../TRDP-unlearning/SD_lora_distil/.env')
os.environ['WANDB_DISABLED'] = "true"
assert len(os.getenv('HF_TOKEN'))>0
#!huggingface-cli login --token ${HF_TOKEN}

from vision_unlearning.utils.logger import get_logger, setup_loggers


logger = get_logger('testbed')
setup_loggers()

In [ ]:
# See also https://colab.research.google.com/drive/1wwSvMc7I8qOAWhCnGafSJJAmEyipcLA2#scrollTo=vMthgElB-sZM
#!pip install nltk


In [ ]:
##############################
# Step 1: prepare datasets
##############################
os.makedirs('assets/datasets/imagenet')
is not of.path.exists('assets/datasets/imagenet/attrann.mat'):
    !wget https://www.image-net.org/data/attrann.mat -O assets/datasets/imagenet/attrann.mat

In [ ]:
data = sio.loadmat("assets/datasets/imagenet/attrann.mat")['attrann']#.keys()

df = pd.DataFrame(
    data[0][0][2],
    index=[e[0][0] for e in data[0][0][0]],
    columns=[e[0] for e in data[0][0][1][0]]
)
df['category'] = pd.Series(df.index, index=df.index).str.split('_').str[0]
df.head()

In [ ]:
# If we wanted to enrich object with category data... eh
#df['hyper1'] = df['category'].apply(lambda x: wn.synset_from_pos_and_offset('n', int(x[1:])).hypernyms()[0].name().split('.')[0])
#df['hyper2'] = df['category'].apply(lambda x: wn.synset_from_pos_and_offset('n', int(x[1:])).hypernyms()[0].hypernyms()[0].name().split('.')[0])
#df['hyper3'] = df['category'].apply(lambda x: wn.synset_from_pos_and_offset('n', int(x[1:])).hypernyms()[0].hypernyms()[0].hypernyms()[0].name().split('.')[0])

#df['category_name'] = df['category'].apply(lambda x: wn.synset_from_pos_and_offset('n', int(x[1:])).name().split('.')[0])
#df['category_name'].value_counts()

#df['hyper3'].value_counts().head(20)

#df[df['hyper3']=='instrumentality'].value_counts('category_name').head(20)

In [ ]:
##############################
# Step 2: attribute inference
##############################
df_categories = df.groupby('category').mean()
print((df_categories>0.1).mean().mean()*100)
print((df_categories>0.5).mean().mean()*100)
print((df_categories>0.8).mean().mean()*100)
df_categories = df_categories>0.3   # At least x% of the instances have this property
len(df_categories)

# enrich with wordnet
def f(x):
    h4 = wn.synset_from_pos_and_offset('n', int(x[1:])).hypernyms()[0].hypernyms()[0].hypernyms()[0].hypernyms()[0]
    l = h4.hypernyms()
    if len(l)>0:
        return l[0].name().split('.')[0]
    else:
        return h4.name().split('.')[0]
df_categories['hyper1'] = pd.Series(df_categories.index, index=df_categories.index).apply(lambda x: wn.synset_from_pos_and_offset('n', int(x[1:])).hypernyms()[0].name().split('.')[0])
df_categories['hyper2'] = pd.Series(df_categories.index, index=df_categories.index).apply(lambda x: wn.synset_from_pos_and_offset('n', int(x[1:])).hypernyms()[0].hypernyms()[0].name().split('.')[0])
df_categories['hyper3'] = pd.Series(df_categories.index, index=df_categories.index).apply(lambda x: wn.synset_from_pos_and_offset('n', int(x[1:])).hypernyms()[0].hypernyms()[0].hypernyms()[0].name().split('.')[0])
df_categories['hyper4'] = pd.Series(df_categories.index, index=df_categories.index).apply(lambda x: wn.synset_from_pos_and_offset('n', int(x[1:])).hypernyms()[0].hypernyms()[0].hypernyms()[0].hypernyms()[0].name().split('.')[0])
df_categories['hyper5'] = pd.Series(df_categories.index, index=df_categories.index).apply(f)

df_categories['category_name'] = pd.Series(df_categories.index, index=df_categories.index).apply(lambda x: wn.synset_from_pos_and_offset('n', int(x[1:])).name().split('.')[0])

In [ ]:
cats = ["placental", "animal", "mammal", "chordate", "vertebrate", "ungulate", "organism", "living_thing", "domestic_animal", "aquatic_mammal", "plant", "bony_fish", "plant_organ", "reptile", "primate", "ruminant", "even-toed_ungulate", "foodstuff", "food"]
df_categories2 = df_categories[
    ~df_categories['hyper5'].isin(cats)
    & ~df_categories['hyper4'].isin(cats)
    & ~df_categories['hyper3'].isin(cats)
    & ~df_categories['hyper2'].isin(cats)
    & ~df_categories['hyper1'].isin(cats)
]

print(df_categories2.shape)
df_categories2.head(100).tail(20)

In [ ]:
cats = ['vehicle', 'artifact', 'object', 'instrumentality', 'vehicle', 'wheeled_vehicle', 'whole']
df_categories2 = df_categories[
    df_categories['hyper5'].isin(cats)
    | df_categories['hyper4'].isin(cats)
    | df_categories['hyper3'].isin(cats)
    | df_categories['hyper2'].isin(cats)
    | df_categories['hyper1'].isin(cats)
]

print(df_categories2.shape)
df_categories2